In [1]:
#%pip install librosa music21 numpy matplotlib soundfile

In [2]:
import librosa
import numpy as np

In [3]:
'''
true pitches: 44, 50, 56
'''
import json
with open("nsynth-train/examples.json", "r") as f:
    data=json.load(f)

samples=[]
true_pitches=[]
true_notes=[]
for key,value in data.items():
    if "keyboard_acoustic" in key:
        if value["pitch"] in [i for i in range(44,57,2)]:
            samples.append(str(key))
            true_pitches.append(value["pitch"])
            true_notes.append(librosa.midi_to_note(value['pitch']))


In [4]:
import os,shutil
os.makedirs("./input", exist_ok=True)

json_path = "nsynth-train/examples.json"
audio_folder = "nsynth-train/audio"
input_folder = "./input"

for file in samples:
    src=os.path.join(audio_folder, file + ".wav")
    dst=os.path.join(input_folder, file + ".wav")

    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        print(f"Missing file: {file}")


In [5]:
import pandas as pd
json_path = "nsynth-train/examples.json"
input = "./input"

pred_pitches=[]
pred_notes=[]
for file in os.listdir(input):
    path = os.path.join(input,file)

    y,sr=librosa.load(path,sr=16000)

    f0 = librosa.yin(y, fmin=50, fmax=500, sr=sr)
    valid = f0[~np.isnan(f0)]
    pred_pitch = np.median(valid)

    pred_pitches.append(pred_pitch)
    pred_notes.append(librosa.hz_to_note(pred_pitch))

res=[]
for i in range(len(samples)):
    res.append([
        samples[i],
        true_pitches[i],
        true_notes[i],
        pred_pitches[i],
        pred_notes[i]
    ])

df=pd.DataFrame(
    res, columns=[
        "sample",
        "true_pitch",
        "true_note",
        "pred_pitch",
        "pred_note"
    ]
)

print(df)


                            sample  true_pitch true_note  pred_pitch pred_note
0    keyboard_acoustic_018-046-100          46       A♯2  207.877498       G♯3
1    keyboard_acoustic_008-054-127          54       F♯3   81.936685        E2
2    keyboard_acoustic_010-048-100          48        C3  146.692939        D3
3    keyboard_acoustic_001-046-050          46       A♯2  116.579131       A♯2
4    keyboard_acoustic_003-050-100          50        D3  130.942439        C3
..                             ...         ...       ...         ...       ...
657  keyboard_acoustic_011-054-075          54       F♯3  104.064915       G♯2
658  keyboard_acoustic_007-056-025          56       G♯3  164.605264        E3
659  keyboard_acoustic_018-052-127          52        E3  111.857390        A2
660  keyboard_acoustic_005-046-127          46       A♯2  115.985229       A♯2
661  keyboard_acoustic_019-052-025          52        E3  146.750807        D3

[662 rows x 5 columns]


In [6]:
df["true_note"] = df["true_note"].str.replace("♯", "#")
df["pred_note"] = df["pred_note"].str.replace("♯", "#")
df["true_note"] = df["true_note"].str.replace("♭", "b")
df["pred_note"] = df["pred_note"].str.replace("♭", "b")
df["Match"] = df["true_note"] == df["pred_note"]
accuracy = df["Match"].mean()
print(f"Accuracy: {accuracy:.2%}")
df.to_csv("results.csv", index=False)

Accuracy: 13.29%


In [7]:
%pip install music21

Note: you may need to restart the kernel to use updated packages.


In [8]:
from music21 import stream, note

s = stream.Stream()

for n_name in df["pred_note"]:
    n = note.Note(n_name)
    n.quarterLength = 1
    s.append(n)

s.write('musicxml', fp='./output/batch_output.xml')

/Users/amishasingh/Documents/sheetgenie/nsynth/.venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


PosixPath('/Users/amishasingh/Documents/sheetgenie/nsynth/output/batch_output.xml')

In [9]:
len(samples)

662